# Data cleaning and preprocessing

In [90]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

STATE = 42

In [91]:
hurricanes = pd.read_csv("../ships/ships_0hr_cleaned.csv") # renamed kaggle csv from storms to hurricane_data

## Check df info

In [92]:
hurricanes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14328 entries, 0 to 14327
Columns: 123 entries, storm_name to PCM3
dtypes: float64(120), object(3)
memory usage: 13.4+ MB


In [93]:
print(len(hurricanes) - len(hurricanes.dropna()))

14213


Almost every row has a missing value, we will now inspect what columns have missing data

In [94]:
# find out what percentage of missing data is in each column
overthou = 0
for col in hurricanes.columns:
    print(col + ": ", len(hurricanes) - len(hurricanes.dropna(subset=[col])))
    if len(hurricanes) - len(hurricanes.dropna(subset=[col])) > 1000:
        overthou += 1
print(overthou)

storm_name:  0
storm_id:  0
datetime:  0
ships_vmax_head:  0
ships_lat_head:  0
ships_long_head:  0
ships_mslp_head:  127
VMAX:  0
MSLP:  127
TYPE:  0
HIST:  0
DELV:  0
INCV:  1922
LAT:  0
LON:  0
CSST:  0
CD20:  0
CD26:  0
COHC:  0
DTL:  1297
OAGE:  0
NAGE:  0
U200:  1331
U20C:  1331
V20C:  1331
E000:  1331
EPOS:  1331
ENEG:  1331
EPSS:  1331
ENSS:  1331
RHLO:  1331
RHMD:  1331
RHHI:  1331
PSLV:  2
Z850:  1331
D200:  1331
REFC:  1331
PEFC:  1331
T000:  1331
R000:  1331
Z000:  1331
TLAT:  1331
TLON:  1331
TWAC:  1506
TWXC:  1506
G150:  1331
G200:  1331
G250:  1331
V000:  1331
V850:  1331
V500:  1331
V300:  1331
TGRD:  1331
TADV:  1331
PENC:  1331
SHDC:  1331
SDDC:  1331
SHGC:  1331
DIVC:  1506
T150:  1331
T200:  1331
T250:  1331
SHRD:  1331
SHTD:  1331
SHRS:  1331
SHTS:  1331
SHRG:  1331
PENV:  1331
VMPI:  1331
VVAV:  1331
VMFX:  1331
VVAC:  1331
HE07:  1331
HE05:  1331
O500:  1331
O700:  1331
CFLX:  1331
MTPW:  0
PW01:  1331
PW02:  1331
PW03:  1331
PW04:  1331
PW05:  1331
PW06:  1331


The column XD30 contains 14108 rows that have missing data. This column will be dropped because the amount of rows that do not have missing data in this column is insignificant in relation to the entire dataset.

In [95]:
hurricanes = hurricanes.drop('XD30', axis=1)

In [96]:
hurricanes['datetime'] = pd.to_datetime(hurricanes['datetime'])

# Filter using the built-in dt accessor
above_year = hurricanes[hurricanes['datetime'].dt.year > 1990].copy()

In [97]:
len(above_year)

12021

In [98]:
max_missing = 0.15 * len(above_year)
above_year_clean = above_year.dropna(thresh=len(above_year) - max_missing, axis=1)

print(f"Original columns: {above_year.shape[1]}")
print(f"Columns after dropping high-sparsity features: {above_year_clean.shape[1]}")

Original columns: 122
Columns after dropping high-sparsity features: 115


In [99]:
print(above_year.isnull().sum())

storm_name            0
storm_id              0
datetime              0
ships_vmax_head       0
ships_lat_head        0
                   ... 
IRM1                767
IRM3                678
PC00                710
PCM1               1620
PCM3                668
Length: 122, dtype: int64


In [103]:
from sklearn.impute import SimpleImputer

above_year_clean = above_year_clean.copy()

numeric_cols = above_year_clean.select_dtypes("float").columns

imputer = SimpleImputer(strategy='median')

above_year_clean.loc[:, numeric_cols] = imputer.fit_transform(above_year_clean[numeric_cols])

In [105]:
print(above_year_clean.isnull().sum())

storm_name         0
storm_id           0
datetime           0
ships_vmax_head    0
ships_lat_head     0
                  ..
XD24               0
XD22               0
XD20               0
XD18               0
XD16               0
Length: 122, dtype: int64


## Check for duplicate values

In [106]:
above_year_clean.duplicated()

2307     False
2308     False
2309     False
2310     False
2311     False
         ...  
14323    False
14324    False
14325    False
14326    False
14327    False
Length: 12021, dtype: bool

We can see that there are no duplicate values after running the `duplicated` function.

In [110]:
pd.unique(above_year_clean['TYPE'])

array([1., 2.])

## Create binary Non/Major target variable

In [17]:
status_hurricane['hurricane_class'] = 0
status_hurricane.loc[status_hurricane['category'] > 2.0, 'hurricane_class'] = 1

status_hurricane.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5100 entries, 70 to 22678
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   name                          5100 non-null   object 
 1   year                          5100 non-null   int64  
 2   month                         5100 non-null   int64  
 3   day                           5100 non-null   int64  
 4   hour                          5100 non-null   int64  
 5   lat                           5100 non-null   float64
 6   long                          5100 non-null   float64
 7   status                        5100 non-null   object 
 8   category                      5100 non-null   float64
 9   wind                          5100 non-null   int64  
 10  pressure                      5100 non-null   int64  
 11  tropicalstorm_force_diameter  5100 non-null   int64  
 12  hurricane_force_diameter      5100 non-null   int64  
 13  hurric

In [18]:
# write df to csv
status_hurricane.to_csv('data/cleaned_data.csv', index=False)

In [19]:
# split into training and testing
train_df, test_df = train_test_split(status_hurricane, test_size=0.2, random_state=STATE)

train_df.to_csv('data/train.csv', index=False)
test_df.to_csv('data/test.csv', index=False)